# 🎬 MiniMax H3 视音频生成系统 · Google Colab 专属部署与云盘直存工作流

> **模型特点**：MiniMax H3（海螺 AI）多模态视音频生成模型，支持原生 2K/768p 24FPS 画面与 32kHz 立体声原生伴音。  
> **配置环境**：Google Colab（GPU: A100，建议开启 High-RAM）  
> **特色功能**：
> - 🚀 **ComfyUI 交互式前端**：支持文生视频（T2V）、图生视频（I2V）、参考生视频（R2V）。
> - 💾 **Google Drive 自动直存**：所有生成的 `.mp4` 视频及音频毫秒级同步到您的个人云端硬盘，断连不丢件。
> - ⚡ **4-Step Turbo-LoRA 极速推理**：A100 单卡 30~50 秒快速出片。
> - 📦 **云盘缓存机制（可选）**：支持将 38GB 模型归档在 Google Drive，下次启动无需重新下载。


### 步骤 1：GPU 与系统环境自检
检查是否已分配到 NVIDIA A100 GPU 以及高内存（High-RAM）环境。


In [ ]:
# @title 1. GPU 与系统资源检查
import torch
import psutil
import os

print("="*60)
print("🔍 正在检查 Google Colab 硬件配置...")
print("="*60)

# 检查 GPU
if not torch.cuda.is_available():
    raise RuntimeError("❌ 未检测到 GPU！请在顶部菜单选择：代码执行程序 -> 更改运行时类型 -> 硬件加速器选择 GPU (A100)。")

gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ GPU 型号: {gpu_name}")
print(f"✅ GPU 显存: {total_vram:.2f} GB")

# 检查系统内存 (RAM)
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"✅ 系统内存 (RAM): {ram_gb:.2f} GB")

if ram_gb < 25:
    print("⚠️ 警告：当前未开启高内存（High-RAM）。建议在运行时设置中勾选 High-RAM 以获得最稳定的体验。")
else:
    print("🎉 恭喜！当前配置完全满足 MiniMax H3 的高速推理要求！")
print("="*60)


### 步骤 2：挂载 Google Drive 并初始化专属存储目录
挂载云端硬盘后，系统将自动创建 `MiniMax_H3_Outputs` 文件夹，并配置可选的模型持久化缓存目录。


In [ ]:
# @title 2. 挂载 Google 云端硬盘
from google.colab import drive
import os

print("📂 正在请求授权挂载 Google Drive...")
drive.mount('/content/drive')

# 定义 Google Drive 路径
GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MiniMax_H3_Outputs"
GDRIVE_MODEL_CACHE = "/content/drive/MyDrive/MiniMax_Models"

os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(GDRIVE_MODEL_CACHE, exist_ok=True)

print("="*60)
print(f"✅ 视频自动保存路径已就绪：{GDRIVE_OUTPUT_DIR}")
print(f"✅ 模型持久化缓存路径已就绪：{GDRIVE_MODEL_CACHE}")
print("="*60)


### 步骤 3：安装 ComfyUI 核心环境与专属导演节点
部署 ComfyUI、ComfyUI-Manager 以及支持 MiniMax H3 多分段镜头控制的 `ComfyUI_MiniMaxH3_Director` 扩展。


In [ ]:
# @title 3. 部署 ComfyUI 及扩展节点
import os
import subprocess

os.chdir('/content')

if not os.path.exists("/content/ComfyUI"):
    print("🚀 正在克隆 ComfyUI 官方代码库...")
    os.system("git clone https://github.com/comfyanonymous/ComfyUI.git")
else:
    print("ℹ️ ComfyUI 代码已存在，跳过克隆。")

os.chdir('/content/ComfyUI')
print("📦 正在安装 PyTorch 与核心加速依赖...")
os.system("pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")
os.system("pip install -q -r requirements.txt")
os.system("pip install -q accelerate diffusers huggingface_hub aria2 sentencepiece")

# 安装插件
os.chdir('/content/ComfyUI/custom_nodes')
if not os.path.exists("ComfyUI-Manager"):
    os.system("git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git")
if not os.path.exists("ComfyUI_MiniMaxH3_Director"):
    os.system("git clone -q https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git")

print("✅ ComfyUI 核心与扩展节点安装完毕！")


### 步骤 4：高速下载 MiniMax H3 官方量化模型（支持 Drive 缓存复用）
下载专为 A100 优化的 FP8 骨干模型、NVFP4 文本/视觉编码器、4-Step Turbo-LoRA 与立体声音频 VAE。  
如果 Google Drive 中已有缓存，将自动建立软链接，无需重复下载 38GB 文件！


In [ ]:
# @title 4. 模型智能下载与挂载（支持云盘秒级复用）
import os
import shutil
from huggingface_hub import hf_hub_download

REPO_ID = "Comfy-Org/MiniMax-H3"
LOCAL_MODELS_DIR = "/content/ComfyUI/models"
GDRIVE_MODEL_CACHE = "/content/drive/MyDrive/MiniMax_Models"

# 是否同步缓存到 Google Drive (True: 备份到云盘下次秒启; False: 仅存临时磁盘)
SAVE_MODELS_TO_GDRIVE = True # @param {type:"boolean"}

target_files = [
    # (子目录, 远程文件名)
    ("diffusion_models", "minimax_h3_fl2va_pruned_fp8_scaled.safetensors"),
    ("text_encoders", "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"),
    ("loras", "minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"),
    ("vae", "minimax_h3_audio_vae_fp32.safetensors"),
    ("vae", "minimax_h3_video_vae_int8_convrot.safetensors")
]

print("="*60)
print("📥 正在处理 MiniMax H3 核心模型权重...")
print("="*60)

for sub_dir, fname in target_files:
    local_sub_path = os.path.join(LOCAL_MODELS_DIR, sub_dir)
    os.makedirs(local_sub_path, exist_ok=True)
    local_file = os.path.join(local_sub_path, fname)
    
    gdrive_sub_path = os.path.join(GDRIVE_MODEL_CACHE, sub_dir)
    os.makedirs(gdrive_sub_path, exist_ok=True)
    gdrive_file = os.path.join(gdrive_sub_path, fname)
    
    # 策略 1: Google Drive 中已有该模型，直接创建软链接（秒级完成）
    if os.path.exists(gdrive_file):
        print(f"⚡ [云盘复用] 检测到 Google Drive 缓存: {fname}")
        if not os.path.exists(local_file):
            os.symlink(gdrive_file, local_file)
        continue
    
    # 策略 2: 下载模型
    if not os.path.exists(local_file):
        print(f"🌐 [高速下载] 正在从 Hugging Face 下载: {fname} ...")
        hf_hub_download(
            repo_id=REPO_ID,
            filename=f"{sub_dir}/{fname}",
            local_dir=LOCAL_MODELS_DIR
        )
    
    # 策略 3: 若勾选备份到云盘，直接复制到 Google Drive 供下次使用
    if SAVE_MODELS_TO_GDRIVE and not os.path.exists(gdrive_file):
        print(f"💾 [云盘备份] 正在将 {fname} 同步备份到 Google Drive ...")
        shutil.copy2(local_file, gdrive_file)

print("="*60)
print("🎉 所有模型配置就绪！")
print("="*60)


### 步骤 5：下载并注入预置工作流模版
将官方经过 A100 + Turbo-LoRA 调优的文生视频（T2V）与图生视频（I2V）工作流注入 ComfyUI。


In [ ]:
# @title 5. 注入优化后的官方工作流模版
import urllib.request
import os

workflows_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(workflows_dir, exist_ok=True)

templates = {
    "minimax_h3_t2v.json": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_t2v.json",
    "minimax_h3_i2v.json": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_i2v.json",
    "minimax_h3_r2v.json": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_r2v.json"
}

for name, url in templates.items():
    dest = os.path.join(workflows_dir, name)
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"✅ 工作流模版已注入: {name}")
    except Exception as e:
        print(f"⚠️ 下载 {name} 失败: {e}")

print("🎨 工作流模版全部注入成功，可在 WebUI 的 Load 菜单中直接选择！")


### 步骤 6：启动 ComfyUI 并连接 Cloudflare 安全公网隧道
服务启动后，ComfyUI 的输出目录将直接绑定到 Google Drive：
`/content/drive/MyDrive/MiniMax_H3_Outputs`  
在下方日志中点击 `https://xxxx.trycloudflare.com` 即可打开前端界面开始生成！


In [ ]:
# @title 6. 启动 ComfyUI Web 服务（直存 Google Drive）
import subprocess
import time
import os

os.chdir('/content/ComfyUI')

# 下载 cloudflared 穿透工具
if not os.path.exists("/tmp/cloudflared"):
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared")
    os.system("chmod +x /tmp/cloudflared")

GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MiniMax_H3_Outputs"

# 启动 ComfyUI 主程序
cmd_comfy = [
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--highvram",
    "--output-directory", GDRIVE_OUTPUT_DIR
]

print("🚀 正在启动 ComfyUI 后台进程...")
p_comfy = subprocess.Popen(cmd_comfy)
time.sleep(5)

print("\n" + "="*70)
print("🎬 MiniMax H3 视音频生成服务已成功启动！")
print(f"📁 自动同步路径: Google Drive -> 我的云端硬盘 -> MiniMax_H3_Outputs")
print("🔗 请点击下方 Cloudflare 临时公网链接访问 Web 界面：")
print("="*70 + "\n")

# 启动 Cloudflare 隧道并实时打印访问网址
subprocess.run(["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"])
